<div style="display:flex; align-items:center; gap:18px; text-align:left">

<div>
<p>Ingeniería en Información y Control de Gestión</p>
<p>Facultad de Economía y Negocios</p>
<p>Introducción a Machine Learning</p>
<p>Proyecto capstone: plantilla de trabajo</p>
</div>
</div>

# Proyecto capstone · plantilla de trabajo

Esta plantilla ordena el proyecto en pasos, en el mismo orden que la presentación del
[enunciado](https://sebastiancontz.github.io/ust-introduccion-machine-learning/ediciones/2026/proyecto-capstone.pdf). Cada paso dice **qué hacer**, qué funciones sirven y en qué sesión se vieron; el
código lo escriben ustedes en la celda que sigue, guiados por las pistas en comentarios.

**Cómo usarla**

1. **Archivo → Guardar una copia en Drive**, y trabajen sobre esa copia.
2. Recorran los pasos en orden. Si un paso necesita más de una celda, agréguenla debajo.
3. Anoten cada decisión y su motivo en la celda de texto del paso: de ahí sale lo que defienden.
4. Los pasos marcados **(opcional)** pueden saltarse.
5. Antes de copiar una cifra a la presentación: **Entorno de ejecución → Reiniciar sesión y ejecutar
   todo**. Solo vale la cifra que sale igual.

## Paso 0 · Librerías

En Colab casi todas vienen instaladas; la primera celda asegura las que falten. La segunda importa todo
lo que se usó en el curso. Si usan algo más, agréguenlo acá.

In [ ]:
%%capture
!pip install -q pandas numpy pyarrow matplotlib seaborn scikit-learn shap joblib

In [ ]:
import os                          # revisa si los datos están en una carpeta local
import time                        # mide cuánto tarda cada modelo
import numpy as np                 # cálculo numérico
import pandas as pd                # tablas
import matplotlib.pyplot as plt    # gráficos
import seaborn as sns              # gráficos estadísticos
import shap                        # explicaciones del modelo
import joblib                      # guarda y carga el modelo
import sklearn                     # para anotar su versión

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.metrics import (roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, mean_absolute_error, silhouette_score)
from sklearn.cluster import KMeans

## Paso 1 · El canvas, antes de programar

Copien acá las respuestas de su canvas: todo lo que viene después tiene que respetarlas.

- **Decisión que apoya el modelo y quién la toma:**
- **Qué es un registro:**
- **Qué se predice y con qué horizonte; clasificación o regresión:**
- **Momento de la predicción (qué se sabe en ese momento y qué no):**
- **Columnas que excluyen y por qué:**
- **Métrica y criterio de éxito (contra qué se comparan):**
- **En clasificación, cuánto cuesta cada error:**

In [ ]:
SEMILLA = 42   # la misma en la muestra, la partición y cada modelo que sortea

# las decisiones del canvas que el código usa, como variables:
# el archivo, el nombre del target, las columnas excluidas y, en clasificación, el costo de los errores

> **Para la presentación:** sección 1 del enunciado: el canvas completo y coherente.

## Paso 2 · Cargar los datos

`BASE` es la carpeta pública con los datasets, la misma del
[Colab de los seis casos](https://colab.research.google.com/github/sebastiancontz/ust-introduccion-machine-learning-colab/blob/main/ediciones/2026/notebooks/capstone-datasets.ipynb). Qué significa cada columna está en el diccionario de datos
del enunciado.

In [ ]:
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO  # en Colab siempre se usa REPO

# carguen su archivo con pd.read_parquet(BASE + 'nombre_del_archivo.parquet') y miren su forma

## Paso 3 · La calidad de los datos

Los archivos **no vienen limpios**. Antes de graficar, cuenten (sesión 3):

- el tipo de cada columna y cuántos vacíos tiene: `.info()`, `.isna().sum()`;
- las filas idénticas a otra: `.duplicated().sum()`;
- los rangos de las numéricas, buscando valores imposibles o extremos: `.describe()`;
- el reparto del target: `.value_counts(normalize=True)` en clasificación, `.describe()` en regresión.

Un vacío también puede venir disfrazado: una categoría `desconocido` o un valor como -1 que el
diccionario explica.

In [ ]:
# tipos y vacíos

# filas repetidas

# rangos de las numéricas

# reparto del target

**Problemas encontrados y cuántos registros afecta cada uno:**

-

> **Para la presentación:** sección 2: registros, columnas y tipos; los problemas de calidad, con cifras.

## Paso 4 · Exploración: 2 o 3 gráficos relevantes

No se trata de graficar todo, sino de elegir los gráficos que **informan una decisión** del proyecto
(sesión 3). Puntos de partida: el reparto del target (`sns.countplot`, `sns.histplot`), una variable
numérica frente al target (`sns.boxplot`, `sns.scatterplot`) y una categórica frente al target. Cada
gráfico lleva un título que diga lo que muestra.

In [ ]:
# gráfico 1

# gráfico 2

# gráfico 3 (si aporta)

**Lectura de cada gráfico** (qué muestra y qué decisión informa):

1.
2.
3.

> **Para la presentación:** sección 2: cada gráfico con lo que muestra y qué decisión informa.

## Paso 5 · Preparación sin fuga

Dos tipos de decisiones, y van en lugares distintos (sesiones 4 y 5):

- **Las que no aprenden de los datos** se aplican acá, sobre todo el archivo: quitar filas repetidas
  (¿son un error o dos registros que coinciden?), excluir las columnas que no se conocen al momento de
  predecir y los identificadores, y tomar una muestra con `.sample(n=..., random_state=SEMILLA)` si el
  archivo es grande para Colab (sesión 13).
- **Las que aprenden de los datos** (la mediana para rellenar, la media para escalar, las categorías
  que existen) van **dentro de un pipeline**, que se ajusta solo con entrenamiento.

Qué hacer:

1. Definir `y` (en clasificación, el sí como 1 y el no como 0) y `X` sin el target ni las excluidas.
2. Separar columnas numéricas y categóricas (`select_dtypes`). Revisen las listas: un código guardado
   como número no es una cantidad, y una fecha no entra tal cual; si sirve algo de ella, créenlo como
   columna aparte.
3. Armar la preparación **sin ajustar** con `ColumnTransformer`: `SimpleImputer` y `StandardScaler` para
   las numéricas, `OneHotEncoder(handle_unknown='ignore')` para las categóricas. Una categórica con
   cientos de valores se vuelve cientos de columnas: decidan si la agrupan, la excluyen o usan una
   muestra.

In [ ]:
# filas repetidas, columnas excluidas y muestra (si la usan)

# X e y

# columnas numéricas y categóricas

# la preparación, sin ajustar

**Decisiones de preparación y su motivo:**

-

> **Para la presentación:** secciones 1 y 2: qué columnas excluyeron y por qué; las decisiones de preparación.

## Paso 6 · La partición: entrenamiento, validación y prueba

Tres partes, con los nombres de la sesión 5: `X_train`/`y_train`, `X_val`/`y_val` y `X_test`/`y_test`
(y `X_trainval`/`y_trainval` para el intermedio). La prueba **no se mira hasta el paso 10**.

La celda trae la partición de la sesión 12: **al azar, 60/20/20, en dos pasos**. Primero se aparta el
20 % de prueba; después, el 25 % de lo que queda va a validación, que es el 20 % del archivo.

- **Si su caso es de clasificación, estratifiquen**: agreguen `stratify=y` en la primera llamada y
  `stratify=y_trainval` en la segunda, como indican los comentarios. Así las tres partes conservan la
  misma proporción del target. En regresión se deja como está: el target es una cantidad, no una
  clase.
- Si los registros están ordenados en el tiempo y la decisión es sobre el futuro (sesión 13), esta
  partición al azar deja que el entrenamiento vea el futuro. En ese caso, reemplácenla: ordenen por
  fecha y corten, para entrenar con el pasado y validar y probar con lo que vino después.

In [ ]:
# primero se aparta la prueba (20 %); lo que queda son entrenamiento y validación todavía juntos
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA)   # en clasificación, agreguen stratify=y

# ese bloque se parte: el 25 % del 80 % que quedó es el 20 % del archivo
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEMILLA)   # en clasificación, agreguen stratify=y_trainval

print('entrenamiento:', len(X_train), '· validación:', len(X_val), '· prueba:', len(X_test))

**Cómo partimos y por qué:**

-

> **Para la presentación:** sección 3: cómo partieron los datos y por qué respeta la estructura de su caso.

## Paso 7 · El baseline y los candidatos

El **baseline va primero**: un modelo que responde siempre lo mismo, `DummyClassifier` o
`DummyRegressor` (sesiones 6 y 7). Después, **al menos 3 algoritmos distintos**; el baseline no cuenta
como uno de ellos. Los del curso:

| Clasificación | Regresión | Sesión |
|:--|:--|:--|
| `LogisticRegression` | `LinearRegression` | 6 y 7 |
| `DecisionTreeClassifier` | `DecisionTreeRegressor` | 8 |
| `RandomForestClassifier` | `RandomForestRegressor` | 8 |
| `HistGradientBoostingClassifier` | `HistGradientBoostingRegressor` | 8 |

Un diccionario `{nombre: modelo}` permite recorrerlos todos con el mismo código (sesión 12).

In [ ]:
# el baseline y los candidatos, con random_state=SEMILLA donde corresponda

> **Para la presentación:** sección 3: el baseline y al menos 3 algoritmos distintos.

## Paso 8 · La comparación, en validación

Cada candidato pasa por **la misma preparación** (`Pipeline` con una copia `clone` de la preparación),
se ajusta **solo con entrenamiento** y se mide en **validación** (sesión 12). Todo en una tabla, con
todas las filas, también las que pierden.

- **Clasificación:** `roc_auc_score` con las probabilidades de `predict_proba` (sesión 7).
- **Regresión:** `mean_absolute_error`, que se nombra completo la primera vez: «el promedio del error
  absoluto, en \<unidad del target\>» (sesiones 6 y 13).
- Midan también en entrenamiento: si es mucho mejor que en validación, el modelo memorizó (sesión 8).
- Midan cuánto tarda cada modelo con `time.time()`: si es demasiado para Colab, usen una muestra.

In [ ]:
# recorrer los candidatos: pipeline, ajuste con entrenamiento y métrica en entrenamiento y validación

# la tabla de comparación

> **Para la presentación:** sección 3: las métricas de todos los modelos en una misma tabla.

## Paso 9 · (opcional) ¿La diferencia es real o es suerte de la partición?

Si dos candidatos quedan cerca, la validación cruzada (*cross-validation*) los mide en varios pliegues
de `X_trainval` y muestra cuánto varía cada uno: `cross_val_score` con `StratifiedKFold` en
clasificación o `KFold` en regresión (sesión 8). Si la diferencia entre medias es menor que esa
variación, elijan por otro motivo. Para ajustar hiperparámetros, `GridSearchCV` sobre el pipeline.

**Si su partición respeta el tiempo, estos pliegues mezclan pasado y futuro**: no los usen así.

In [ ]:
# validación cruzada de los candidatos que quedaron cerca

## Paso 10 · La elección, antes de abrir la prueba

Elegir no es tomar la cifra más alta: pesen el costo de los errores, la simplicidad y si la diferencia
supera la variación (sesión 12). **Escriban el motivo antes de abrir la prueba.**

En clasificación, elijan también el **umbral** en validación (sesión 7): recorran una grilla de
umbrales, cuenten con `confusion_matrix` los positivos que se escapan y las falsas alarmas de cada uno,
y quédense con el de menor costo según el canvas. En regresión no hay umbral que elegir.

In [ ]:
# el modelo elegido

# en clasificación: el costo de cada umbral en validación y el umbral elegido

**Elegimos** … **porque** … 

**Umbral** (solo clasificación): … **porque** …

> **Para la presentación:** sección 3: cuál eligieron y por qué; en clasificación, también con qué umbral.

## Paso 11 · El cierre con prueba, una sola vez

Con el modelo y el umbral ya elegidos, se abre la prueba **una vez** (sesiones 8 y 12): la métrica del
elegido y la del baseline sobre `X_test`, y en clasificación la matriz de confusión con su umbral
(`ConfusionMatrixDisplay.from_predictions`). Si después de ver esta cifra cambian de modelo o de umbral,
la prueba deja de ser una prueba: repórtenlo así.

In [ ]:
# el elegido y el baseline, medidos en prueba

> **Para la presentación:** secciones 3 y 5: el resultado final, frente al baseline y al criterio de éxito.

## Paso 12 · Interpretación con SHAP

Tres miradas (sesión 10): **global** (`shap.plots.beeswarm`), **un caso concreto**
(`shap.plots.waterfall`) y **dependencia** de una variable (`shap.plots.scatter`).

Qué hacer:

1. Tomar una muestra de prueba (unos cientos de registros: con miles, SHAP tarda mucho) y pasarla por
   la preparación del pipeline elegido: `modelo.named_steps['preparar'].transform(...)`.
2. Crear el explicador sobre el modelo, no sobre el pipeline: `shap.TreeExplainer` para árboles,
   bosques y boosting; `shap.LinearExplainer` para un modelo lineal.
3. En un bosque de clasificación la explicación trae las dos clases: tomen la del sí con `[:, :, 1]`.

Para ver los nombres de las columnas en los gráficos, la preparación tiene que devolver una tabla:
`.set_output(transform='pandas')` en el `ColumnTransformer` y `sparse_output=False` en el
`OneHotEncoder`. Si escalaron las numéricas, los ejes muestran valores estandarizados: para verlos en
sus unidades, reemplacen `explicacion.data` por los valores originales, como en los ejemplos resueltos.
SHAP explica **el modelo, no el mundo**: muestra asociaciones, no causas.

In [ ]:
# muestra de prueba, preparada

# explicador y explicación

# global

# un caso concreto

# dependencia de una variable

**Qué permiten afirmar estas explicaciones y qué no:**

-

> **Para la presentación:** sección 4: la explicación global, la de un caso y la de dependencia, leídas como asociación.

## Paso 13 · De la cifra a la recomendación

Este paso no tiene código: es lo que el equipo concluye con lo que salió de los pasos anteriores.

| Pregunta | Respuesta del equipo |
|:--|:--|
| ¿Cuánto mejora el modelo frente al baseline, en la unidad de la decisión? | |
| ¿Se cumplió el criterio de éxito del canvas? ¿Qué cambió respecto de lo planificado y por qué? | |
| ¿Qué gana la organización al usar el modelo frente a no usarlo? | |
| ¿Qué riesgos y límites tiene (sesgos, privacidad, datos de otra época o de otro lugar)? | |
| Si el modelo se usara, ¿qué habría que observar para saber que sigue funcionando? | |

> **Para la presentación:** sección 5: el resultado frente al baseline y al canvas, el valor para la
> organización, los riesgos y límites, y qué observar.

## Paso 14 · (opcional) Segmentación con K-means

Como en la sesión 9: con variables numéricas que describan a los registros de entrenamiento,

1. estandarizar con `StandardScaler`, para que ninguna domine la distancia por su escala;
2. probar varios `k` con `KMeans` y comparar la inercia (método del codo) y `silhouette_score` (con
   `sample_size` si son muchos registros);
3. perfilar cada grupo con el promedio de sus variables (`groupby`) y ponerle un nombre.

Los grupos valen si cada uno habilita una acción distinta para la decisión.

In [ ]:
# variables y estandarización

# inercia y silhouette para varios k

# perfil de cada grupo

**Nombre y acción de cada grupo:**

-

> **Para la presentación:** sección 6, optativa: grupos perfilados, nombrados y útiles para la decisión.

## Paso 15 · (opcional) Exportar el modelo y los resultados

Como en la sesión 12: `joblib.dump` guarda el pipeline ya ajustado y `joblib.load` lo recupera sin
reentrenar; comprueben que el cargado predice lo mismo. Dos cuidados: carguen solo archivos propios o
de confianza, y con las mismas versiones de las librerías (anótenlas: `pd.__version__`,
`sklearn.__version__`…). En clasificación, el pipeline no guarda el umbral: anótenlo aparte.

Para llevar la tabla a la presentación, `.to_csv(...)`; para bajar un archivo de Colab,
`from google.colab import files` y `files.download(...)`.

In [ ]:
# guardar, cargar y comprobar el pipeline

# versiones de las librerías

## Antes de llevar cifras a la presentación

- [ ] **Entorno de ejecución → Reiniciar sesión y ejecutar todo**, y el notebook corre de arriba abajo
      sin errores.
- [ ] Las cifras de la presentación son las de esa corrida.
- [ ] La semilla, la muestra (si usaron una: cuántos registros y con qué semilla) y las versiones de las
      librerías están anotadas.
- [ ] La prueba se abrió una sola vez, después de elegir modelo y umbral.
- [ ] Cada decisión tiene su motivo escrito en el paso que corresponde.

## Atribución de datos

Los seis archivos son obras derivadas: se tradujeron columnas y categorías al español y se resolvieron
tipos, sin limpiar (readmisión conserva una hospitalización por paciente). El detalle de cada
modificación está en el anexo del [enunciado del proyecto](https://sebastiancontz.github.io/ust-introduccion-machine-learning/ediciones/2026/proyecto-capstone.pdf).

- **Telemarketing bancario.** **Creador:** S. Moro, P. Rita y P. Cortez. **Fuente:** [UCI, *Bank Marketing*](https://archive.ics.uci.edu/dataset/222/bank+marketing). **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). **Modificación:** columnas y categorías en español; mes como número.
- **Compras en línea.** **Creador:** C. Sakar e Y. Kastro. **Fuente:** [UCI, *Online Shoppers Purchasing Intention Dataset*](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset). **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). **Modificación:** columnas y categorías en español; mes como número.
- **Readmisión hospitalaria.** **Creador:** J. Clore, K. Cios, J. DeShazo y B. Strack. **Fuente:** [UCI, *Diabetes 130-US Hospitals for Years 1999-2008*](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008). **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). **Modificación:** una hospitalización por paciente, la primera; columnas y categorías en español; target convertido en sí/no (readmisión antes de 30 días).
- **Bicicletas compartidas.** **Creador:** H. Fanaee-T. **Fuente:** [UCI, *Bike Sharing*](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset). **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). **Modificación:** columnas y categorías en español; clima devuelto a sus unidades; sin la columna de índice.
- **Cancelaciones hoteleras.** **Creador:** N. António, A. de Almeida y L. Nunes. **Fuente:** [*Hotel booking demand datasets*, Data in Brief 22 (2019)](https://doi.org/10.1016/j.dib.2018.11.126), material suplementario. **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). **Modificación:** los dos hoteles en un archivo, con la columna `hotel`; columnas y categorías en español; mes como número.
- **Netbilling.** **Creador:** Comisión Nacional de Energía. **Fuente:** [datos.gob.cl, *Netbilling*](https://datos.gob.cl/dataset/netbilling). **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). **Modificación:** potencia y fechas convertidas a número y fecha.